# Phase 3: more recommendation data (Google Colab)

Continues from the phase-2 model with **4x more `next_movie` examples**: 20 target positions per user instead of 5
(~118,000 recommendation samples instead of ~30,000), 4 epochs, metadata tasks still in the mix. The question it
answers: does the gap to SASRec (which learns from every position of every history) come from training data?

Validation is again 500 held-out users (their 2nd-to-last rating) and the checkpoint with the lowest **Next Movie
Loss** is kept at the end. Drawing 20 targets per user changes the random draws, so these are mostly *different*
users from phase 2's: compare the loss with phase 2's 0.688 only roughly. The real comparison is the test evaluation
(`eval_qwen_colab.ipynb`, the same 1,000 users as every other model).

Expect roughly 6-7 hours on an L4-class GPU (~16,000 steps) - **keep this tab open**: checkpoints live on the Colab
disk and a runtime reset loses the progress.

| task | question | answer |
|---|---|---|
| `plot` | What is the movie "*title*" about? | Wikipedia plot (first `plot_words` words) |
| `genres` | Which genres does "*title*" belong to? | MovieLens genres |
| `details` | Who made "*title*" and who stars in it? | director + main cast |
| `identify` | Which movie is this? *plot snippet* | title (year) |
| `next_movie` *(optional)* | A user rated these movies … which next? | title (year) |

**Before you run:**
1. **Runtime → Change runtime type → GPU** (L4 / A100; a T4 works but is slower).
2. On Google Drive you need `MyDrive/GenRec/ml-1m/movies_wiki.csv`, `MyDrive/GenRec/ml-1m/ratings.dat` and the
   phase-2 model in `MyDrive/GenRec/qwen2.5-0.5b-movielens-rec/final`.
3. **Runtime → Run all.** The model is saved to `MyDrive/GenRec/qwen2.5-0.5b-movielens-rec-v2/final` (section 10).
   Evaluate it with `eval_qwen_colab.ipynb` (set `MODEL_NAME = "qwen2.5-0.5b-movielens-rec-v2"`).

After a `CUDA out of memory` error, use **Runtime → Restart session** before running again: the crashed run keeps
its GPU memory until then.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# versions the code was written and tested against (transformers v5 API)
%pip install -q transformers==5.18.0 peft==0.21.1 accelerate==1.15.0

In [ ]:
from pathlib import Path

USE_DRIVE = True   # False: upload movies_wiki.csv each session instead (lost when the runtime resets)
DRIVE_DIR = Path("/content/drive/MyDrive/GenRec")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DIR / "ml-1m"
    START_DIR = DRIVE_DIR / "qwen2.5-0.5b-movielens-rec" / "final"   # the phase-2 model
    FINAL_DIR = DRIVE_DIR / "qwen2.5-0.5b-movielens-rec-v2" / "final"
else:
    from google.colab import files
    DATA_DIR = Path("/content/ml-1m")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():   # pick movies_wiki.csv and ratings.dat
        (DATA_DIR / name).write_bytes(data)
    START_DIR = Path("/content/qwen2.5-0.5b-movielens-rec/final")   # the phase-2 run in this same session
    FINAL_DIR = Path("/content/qwen2.5-0.5b-movielens-rec-v2/final")

# checkpoints stay on the Colab disk: a full fine-tune checkpoint (weights + optimizer) is ~6 GB
CHECKPOINT_DIR = Path("/content/checkpoints/qwen2.5-0.5b-movielens-rec-v2")
assert (DATA_DIR / "movies_wiki.csv").exists(), f"upload movies_wiki.csv to {DATA_DIR}"
assert (DATA_DIR / "ratings.dat").exists(), f"upload ratings.dat to {DATA_DIR}"
assert (START_DIR / "config.json").exists(), (
    f"no phase-2 model in {START_DIR}: run train_qwen_colab_phase2.ipynb first (incl. the save cell)")
print("data:", sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
import json
import os
import random
import re
import time
from dataclasses import dataclass, field, asdict

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation; before torch

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> T4 GPU"
METADATA_TASKS = ["plot", "genres", "details", "identify"]
ALL_TASKS = METADATA_TASKS + ["next_movie"]

## 1. Config

In [ ]:
RUN_TRAINING = True   # False: build data + model and run the checks only


@dataclass
class Config:
    model: str = str(START_DIR)    # continue from the phase-2 model
    data_dir: Path = DATA_DIR
    output_dir: Path = CHECKPOINT_DIR
    tasks: list = field(default_factory=lambda: METADATA_TASKS + ["next_movie"])   # metadata too: no forgetting
    # data
    plot_words: int = 40         # plot answers: a 1-2 sentence summary (150 words is hard to memorise)
    snippet_words: int = 80      # plot snippet length for the identify task
    max_cast: int = 5
    history_len: int = 20        # next_movie: movies shown in the prompt
    min_history: int = 5         # next_movie: minimum history before a target
    next_per_user: int = 20      # next_movie: training targets per user (phase 2: 5)
    next_eval_users: int = 500   # next_movie: users in the eval set
    eval_frac: float = 0.0       # all metadata is trained on; the held-out next_movie users are the validation
    max_length: int = 600        # 20-movie histories reach ~510 tokens
    # training
    epochs: float = 4            # 4 passes over ~4x more recommendation data than phase 2
    batch_size: int = 4          # next_movie prompts are long (~400 tokens); the memory check tells you if it fits
    grad_accum: int = 8          # effective batch = batch_size * grad_accum = 32
    lr: float = 2e-5             # lower than phase 1: refine the model, don't re-learn it
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    eval_steps: int = 1000       # ~16,000 steps in total: evaluating every 250 would add ~45 min
    save_steps: int = 1000       # must equal eval_steps to keep the best checkpoint
    logging_steps: int = 10
    group_by_length: bool = True  # batch samples of similar length: much less padding
    gradient_checkpointing: bool = False   # only needed if you hit CUDA out-of-memory
    lora: bool = False           # a 16 GB GPU fits the full fine-tune; True = LoRA (~9M trainable params, faster)
    lora_r: int = 16
    lora_alpha: int = 32
    seed: int = 42

    def __post_init__(self):
        unknown = set(self.tasks) - set(ALL_TASKS)
        assert not unknown, f"unknown tasks {sorted(unknown)}; choose from {ALL_TASKS}"
        if self.lr is None:
            self.lr = 2e-4 if self.lora else 2e-5


cfg = Config()
set_seed(cfg.seed)
cfg

## 2. Build the samples

In [ ]:
SYSTEM = "You are a movie expert who knows the MovieLens catalogue and recommends movies."
ARTICLES = r"The|A|An|La|Le|Les|L'|Il|El|Das|Der|Die"
ABBREVIATIONS = {"st", "dr", "mr", "mrs", "ms", "jr", "sr", "vs", "lt", "sgt", "capt", "col", "gen", "no"}
END = "<|im_end|>"

TEMPLATES = {
    "plot": ['What is the movie "{t}" about?', 'Summarize the plot of "{t}".', 'Tell me the story of the movie "{t}".'],
    "genres": ['Which genres does the movie "{t}" belong to?', 'What kind of movie is "{t}"?', 'List the genres of "{t}".'],
    "details": ['Who made the movie "{t}" and who stars in it?', 'Who directed "{t}" and who is in the cast?'],
    "identify": ["Which movie is this?\n\n{p}", "Name the movie with this plot:\n\n{p}"],
}


def display_title(ml_title):
    """'City of Lost Children, The (Cité des enfants perdus, La) (1995)' -> 'The City of Lost Children (1995)'"""
    m = re.match(r"^(.*?)\s*\((\d{4})\)\s*$", ml_title)
    name, year = (m.group(1), m.group(2)) if m else (ml_title, None)
    name = re.sub(r"\s*\([^()]*\)", "", name).strip() or name   # drop alternate titles
    name = re.sub(rf"^(.*), ({ARTICLES})$",
                  lambda a: a[2] + ("" if a[2].endswith("'") else " ") + a[1], name)
    return f"{name} ({year})" if year else name


def clean_plot(text):
    text = re.sub(r"\[\d+\]", "", str(text))   # citation markers
    return re.sub(r"\s+", " ", text).strip()


def truncate_words(text, n):
    """Cut to ~n words, ending on the last full sentence when that keeps at least half the text."""
    words = text.split()
    if len(words) <= n:
        return text
    cut = " ".join(words[:n])
    ends = [m.end() for m in re.finditer(r"[.!?][\"')]?(?=\s)", cut)
            if cut[:m.start()].split()[-1].lower().strip("\"'(") not in ABBREVIATIONS]
    return cut[:ends[-1]] if ends and ends[-1] > len(cut) * 0.5 else cut + " ..."


def mask_title(text, display):
    """Hide the movie's own name in an identify snippet so the answer isn't given away."""
    name = re.sub(r"\s*\(\d{4}\)$", "", display)
    for n in sorted({name, re.sub(r"^(The|A|An) ", "", name)}, key=len, reverse=True):
        if len(n) >= 4:   # skip very short names like "It" / "Big" that are ordinary words
            text = re.sub(rf"\b{re.escape(n)}\b", "___", text)
    return text


def is_known(value):
    return isinstance(value, str) and value.strip() and value.strip().lower() != "unknown"


def load_movies(data_dir):
    movies = pd.read_csv(data_dir / "movies_wiki.csv")
    movies["display"] = movies["Title"].map(display_title)
    # two different movies must never share a display title (it is the answer for identify / next_movie)
    dup = movies["display"].duplicated(keep=False)
    movies.loc[dup, "display"] = movies.loc[dup, "Title"]
    assert movies["display"].is_unique, movies.loc[movies["display"].duplicated(keep=False), "display"]
    return movies

In [ ]:
def sample(task, prompt, answer):
    return {"task": task, "prompt": prompt, "answer": answer}


def metadata_samples(movies, tasks, cfg, rng):
    out = []
    for row in movies.to_dict("records"):
        t = row["display"]
        plot = clean_plot(row["Plot"]) if isinstance(row["Plot"], str) else None
        pick = lambda task: rng.choice(TEMPLATES[task])
        if "genres" in tasks:
            out.append(sample("genres", pick("genres").format(t=t), row["Genres"].replace("|", ", ")))
        if "plot" in tasks and plot:
            out.append(sample("plot", pick("plot").format(t=t), truncate_words(plot, cfg.plot_words)))
        if "identify" in tasks and plot:
            snippet = mask_title(truncate_words(plot, cfg.snippet_words), t)
            out.append(sample("identify", pick("identify").format(p=snippet), t))
        if "details" in tasks:
            parts = []
            if is_known(row["Director"]):
                parts.append(f"Directed by {row['Director'].strip()}.")
            if is_known(row["Cast"]):
                cast = [c.strip() for c in re.split(r",|\n", row["Cast"]) if c.strip()][: cfg.max_cast]
                parts.append(f"Starring {', '.join(cast)}.")
            if parts:
                out.append(sample("details", pick("details").format(t=t), " ".join(parts)))
    return out


def next_movie_prompt(history, display, genres):
    lines = [f"{k}. {display[m]} - {genres[m]}" for k, m in enumerate(history, 1)]
    return ("Here are the movies a user rated, oldest first:\n" + "\n".join(lines)
            + "\n\nWhich movie will they rate next? Answer with the title and year.")


def next_movie_samples(data_dir, movies, cfg, rng):
    """Leave-last-out split: train targets come from items[:-2]; eval target is items[-2]; items[-1] is never used."""
    ratings = pd.read_csv(data_dir / "ratings.dat", sep="::", engine="python",
                          names=["UserID", "MovieID", "Rating", "Timestamp"])
    seqs = ratings.sort_values(["UserID", "Timestamp"], kind="stable").groupby("UserID").MovieID.apply(list)
    display = dict(zip(movies.MovieID, movies.display))
    genres = dict(zip(movies.MovieID, movies.Genres.str.replace("|", ", ")))

    train, val = [], []
    for items in seqs:
        train_items = items[:-2]
        positions = list(range(cfg.min_history, len(train_items)))
        for t in rng.sample(positions, min(cfg.next_per_user, len(positions))):
            history = train_items[max(0, t - cfg.history_len):t]
            train.append(sample("next_movie", next_movie_prompt(history, display, genres), display[train_items[t]]))
        history = train_items[-cfg.history_len:]
        val.append(sample("next_movie", next_movie_prompt(history, display, genres), display[items[-2]]))
    rng.shuffle(val)
    return train, val[: cfg.next_eval_users]


def build_samples(cfg):
    rng = random.Random(cfg.seed)
    movies = load_movies(cfg.data_dir)
    meta = metadata_samples(movies, cfg.tasks, cfg, rng)
    rng.shuffle(meta)
    n_eval = int(len(meta) * cfg.eval_frac) if meta else 0
    train, evals = meta[n_eval:], {}
    if n_eval:
        evals["metadata"] = meta[:n_eval]
    if "next_movie" in cfg.tasks:
        nm_train, nm_val = next_movie_samples(cfg.data_dir, movies, cfg, rng)
        train += nm_train
        evals["next_movie"] = nm_val
    rng.shuffle(train)
    return train, evals

In [ ]:
train_samples, eval_samples = build_samples(cfg)
print("train:", pd.Series([s["task"] for s in train_samples]).value_counts().to_dict())
print("eval: ", {k: len(v) for k, v in eval_samples.items()})
pd.DataFrame(train_samples).groupby("task").head(1).set_index("task")

## 3. Tokenize (loss on the answer only)

In [ ]:
def prompt_text(tokenizer, prompt):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


class ChatDataset(Dataset):
    """Tokenized chat samples; labels are -100 on the system/user part so loss covers only the answer."""

    def __init__(self, samples, tokenizer, max_length):
        prompts = tokenizer([prompt_text(tokenizer, s["prompt"]) for s in samples], add_special_tokens=False).input_ids
        answers = tokenizer([s["answer"] + END for s in samples], add_special_tokens=False).input_ids
        self.items, self.dropped = [], 0
        for s, p, a in zip(samples, prompts, answers):
            if len(p) >= max_length - 8:   # prompt alone fills the context: nothing left to learn from
                self.dropped += 1
                continue
            ids = (p + a)[:max_length]
            labels = ([-100] * len(p) + a)[:max_length]
            self.items.append({"input_ids": ids, "labels": labels, "task": s["task"]})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        item = self.items[i]
        return {"input_ids": item["input_ids"], "labels": item["labels"]}


class PadCollator:
    """Pads to a multiple of `multiple` (multiples of 8 suit the GPU's tensor cores)."""

    def __init__(self, pad_id, multiple=8):
        self.pad_id = pad_id
        self.multiple = multiple

    def __call__(self, batch):
        n = max(len(b["input_ids"]) for b in batch)
        n = -(-n // self.multiple) * self.multiple
        pad = lambda seq, value: seq + [value] * (n - len(seq))
        return {
            "input_ids": torch.tensor([pad(b["input_ids"], self.pad_id) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
        }


def describe(name, ds):
    lengths = sorted(len(x["input_ids"]) for x in ds.items)
    tasks = pd.Series([x["task"] for x in ds.items]).value_counts().to_dict()
    print(f"{name}: {len(ds)} samples (dropped {ds.dropped}), tokens median {lengths[len(lengths) // 2]}, "
          f"p95 {lengths[int(len(lengths) * .95)]}, max {lengths[-1]}, tasks {tasks}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model)
train_ds = ChatDataset(train_samples, tokenizer, cfg.max_length)
eval_ds = {k: ChatDataset(v, tokenizer, cfg.max_length) for k, v in eval_samples.items()}
describe("train", train_ds)
for k, v in eval_ds.items():
    describe(f"eval/{k}", v)

In [ ]:
# check the masking: the full chat vs. the part that is actually in the loss, one example per task
first = {}
for i, item in enumerate(train_ds.items):
    first.setdefault(item["task"], i)
for task, i in first.items():
    item = train_ds.items[i]
    answer_ids = [t for t in item["labels"] if t != -100]
    print(f"--- {task} ({len(item['input_ids'])} tokens, {len(answer_ids)} in loss)")
    print(tokenizer.decode(item["input_ids"]))
    print(f"[LOSS ON] {tokenizer.decode(answer_ids)!r}\n")

## 4. Load the model

In [ ]:
device = "cuda"
# T4 (compute capability 7.5) has no fast bf16 -> fp16 mixed precision; L4 / A100 (8.0+) -> bf16
precision = "bf16" if torch.cuda.get_device_capability()[0] >= 8 else "fp16"


def load_model(cfg):
    # fp32 master weights; the Trainer runs the forward/backward in fp16/bf16
    model = AutoModelForCausalLM.from_pretrained(cfg.model, dtype=torch.float32)
    model.config.use_cache = False
    if cfg.lora:
        from peft import LoraConfig, get_peft_model
        model = get_peft_model(model, LoraConfig(
            r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
        if cfg.gradient_checkpointing:
            model.enable_input_require_grads()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"model {cfg.model}: {total / 1e6:.0f}M params, {trainable / 1e6:.1f}M trainable, "
          f"{torch.cuda.get_device_name()}, {precision} mixed precision")
    return model


@torch.no_grad()
def show_generations(model, tokenizer, samples, device, max_new_tokens=80):
    model.eval()
    for s in samples:
        enc = tokenizer(prompt_text(tokenizer, s["prompt"]), return_tensors="pt", add_special_tokens=False).to(device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             eos_token_id=tokenizer.convert_tokens_to_ids(END), pad_token_id=tokenizer.pad_token_id)
        pred = tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"--- [{s['task']}] {s['prompt'][:120]!r}\n    expected:  {s['answer'][:120]!r}\n    generated: {pred[:120]!r}")

model = load_model(cfg).to(device)
collator = PadCollator(tokenizer.pad_token_id)

## 5. Sanity check before training

In [ ]:
batch = {k: v.to(device) for k, v in collator([train_ds[i] for i in range(cfg.batch_size)]).items()}
t0 = time.time()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16 if precision == "bf16" else torch.float16):
    loss = model(**batch).loss
print(f"forward pass on one batch {tuple(batch['input_ids'].shape)} in {precision}: loss {loss.item():.3f} "
      f"({time.time() - t0:.2f}s)")
if not torch.isfinite(loss):   # fp16 can overflow on some models: fall back to full precision
    precision = "fp32"
    print("loss is not finite in fp16 -> training in fp32 (slower)")

In [ ]:
# what the base model answers before fine-tuning
picks = [next(s for s in train_samples if s["task"] == t) for t in first]
picks += [v[0] for v in eval_samples.values()]
show_generations(model, tokenizer, picks, device, max_new_tokens=40)

## 6. Trainer

In [ ]:
class AnswerOnlyTrainer(Trainer):
    """Same loss as the default Trainer, but the 151k-vocab output layer only runs on answer tokens.

    The default computes fp32 logits for every token (~700 MB per copy for a long batch) although only the answer
    is in the loss. Only computing it for the answer saves memory and ~1/4 of the compute.
    """

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        lm = model.get_base_model() if hasattr(model, "get_base_model") else model
        targets = inputs["labels"][:, 1:]
        keep = targets != -100
        with self.accelerator.autocast():   # calling the inner modules skips accelerate's fp16/bf16 forward wrapper
            hidden = lm.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).last_hidden_state
            logits = lm.lm_head(hidden[:, :-1][keep]).float()
        loss = torch.nn.functional.cross_entropy(logits, targets[keep], reduction="sum")
        # num_items_in_batch = answer tokens over all accumulated micro-batches (how the default loss normalises)
        loss = loss / (num_items_in_batch if num_items_in_batch is not None else keep.sum())
        return (loss, {"logits": logits}) if return_outputs else loss


steps_per_epoch = -(-len(train_ds) // (cfg.batch_size * cfg.grad_accum))
print(f"{steps_per_epoch} optimizer steps/epoch x {cfg.epochs} epochs "
      f"(effective batch {cfg.batch_size * cfg.grad_accum}, lr {cfg.lr}, {precision})")

# phase 2: keep the checkpoint that predicts held-out users best (lowest Next Movie Loss)
best_metric = "eval_next_movie_loss" if "next_movie" in eval_ds else None

training_args = TrainingArguments(
    output_dir=str(cfg.output_dir),
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    lr_scheduler_type="cosine",
    warmup_steps=cfg.warmup_ratio,   # transformers v5: a float in [0, 1) is a ratio of total steps
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    eval_strategy="steps" if eval_ds else "no",
    eval_steps=cfg.eval_steps,
    save_strategy="steps",
    save_steps=cfg.save_steps,
    save_total_limit=2,              # the best and the latest checkpoint
    load_best_model_at_end=best_metric is not None,
    metric_for_best_model=best_metric,
    fp16=precision == "fp16",
    bf16=precision == "bf16",
    gradient_checkpointing=cfg.gradient_checkpointing,
    train_sampling_strategy="group_by_length" if cfg.group_by_length else "random",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    seed=cfg.seed,
)
trainer = AnswerOnlyTrainer(model=model, args=training_args, train_dataset=train_ds,
                            eval_dataset=eval_ds or None, data_collator=collator, processing_class=tokenizer)
trainer.create_optimizer()   # catches optimizer / device issues without taking a step
print(f"optimizer: {type(trainer.optimizer).__name__}")

# the answer-only loss must match the model's own loss (eval mode: no LoRA dropout; same precision for both)
model.eval()
with torch.no_grad(), trainer.accelerator.autocast():
    check = {k: v.to(device) for k, v in collator([train_ds[i] for i in range(cfg.batch_size)]).items()}
    ours, ref = trainer.compute_loss(model, check).item(), model(**check).loss.item()
print(f"loss check: answer-only {ours:.4f} vs default {ref:.4f}")
assert abs(ours - ref) < 1e-2   # fp16/bf16 rounding differs slightly between the two

In [ ]:
# memory check: one forward + backward on the longest samples (the trainer starts with those too),
# plus the AdamW state that is only allocated at the first optimizer step
longest = sorted(range(len(train_ds)), key=lambda i: len(train_ds.items[i]["input_ids"]))[-cfg.batch_size:]
batch = {k: v.to(device) for k, v in collator([train_ds[i] for i in longest]).items()}
model.train()
torch.cuda.reset_peak_memory_stats()
trainer.compute_loss(model, batch).backward()
model.zero_grad(set_to_none=True)
adam_gb = 2 * 4 * sum(p.numel() for p in model.parameters() if p.requires_grad) / 2**30
need = torch.cuda.max_memory_allocated() / 2**30 + adam_gb
have = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"memory: ~{need:.1f} GB needed to train (incl. {adam_gb:.1f} GB optimizer state), GPU has {have:.1f} GB")
del batch
torch.cuda.empty_cache()
assert need < 0.85 * have, ("won't fit: halve cfg.batch_size and double cfg.grad_accum, "
                            "or set cfg.lora = True, or cfg.gradient_checkpointing = True")

## 7. Train

Checkpoints go to the Colab disk every `save_steps`. If the cell is interrupted, re-running it resumes from the
last checkpoint, as long as the runtime hasn't been reset.

In [ ]:
if RUN_TRAINING:
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    with open(cfg.output_dir / "run_config.json", "w") as f:
        json.dump({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, f, indent=2)
    resume = any(cfg.output_dir.glob("checkpoint-*"))
    trainer.train(resume_from_checkpoint=True if resume else None)
else:
    print("RUN_TRAINING is False - skipping training")

## 8. After training

In [ ]:
if RUN_TRAINING:
    if eval_ds:
        print(trainer.evaluate())
    log = pd.DataFrame(trainer.state.log_history)
    display(log.dropna(subset=["loss"])[["step", "loss"]].tail())
    show_generations(model, tokenizer, picks, device)

## 9. Recall and recommendation accuracy

Exact-match accuracy on movies the model was trained on (does it know the catalogue?) and, in phase 2, how often its
first guess is the held-out user's next movie (hit@1; a few percent is normal with ~3,700 movies to choose from).

In [ ]:
@torch.no_grad()
def answer(prompt, max_new_tokens=60):
    model.eval()
    enc = tokenizer(prompt_text(tokenizer, prompt), return_tensors="pt", add_special_tokens=False).to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         eos_token_id=tokenizer.convert_tokens_to_ids(END), pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()


def director(text):
    return text.split("Directed by ")[1].split(". Starring")[0].rstrip(".").strip() if "Directed by " in text else None


def is_hit(task, pred, gold):
    if task == "genres":
        return {g.strip() for g in pred.split(",")} == {g.strip() for g in gold.split(",")}
    if task in ("identify", "next_movie"):
        return pred == gold
    return director(pred) == director(gold)   # details: director only (cast order varies)


def recall(samples, n=100, seed=0):
    rng, rows = random.Random(seed), []
    for task in ["genres", "details", "identify"]:
        pool = [s for s in samples if s["task"] == task and (task != "details" or "Directed by" in s["answer"])]
        for s in rng.sample(pool, min(n, len(pool))):
            rows.append({"task": task, "hit": is_hit(task, answer(s["prompt"]), s["answer"])})
    return pd.DataFrame(rows).groupby("task").hit.mean()


if RUN_TRAINING:   # a few minutes: ~300-500 answers generated one by one
    if "next_movie" in eval_samples:
        nm = eval_samples["next_movie"][:200]
        hit1 = sum(is_hit("next_movie", answer(s["prompt"], 30), s["answer"]) for s in nm) / len(nm)
        print(f"next_movie hit@1 on {len(nm)} held-out users: {hit1:.1%}")
    results = {"trained on": recall(train_samples)}
    if "metadata" in eval_samples:
        results["held out"] = recall(eval_samples["metadata"])
    display(pd.DataFrame(results).style.format("{:.0%}"))

## 10. Save the model

In [ ]:
if RUN_TRAINING:
    final = model.merge_and_unload() if cfg.lora else model   # LoRA: fold the adapters into the base weights
    final = final.to(torch.bfloat16)                          # Qwen's native dtype; halves the size (~1 GB)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    final.save_pretrained(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)
    print(f"saved to {FINAL_DIR}")

### Use the model later

Copy `FINAL_DIR` from Google Drive to your machine (e.g. into `checkpoints/`), then:

```python
tokenizer = AutoTokenizer.from_pretrained(final_dir)
model = AutoModelForCausalLM.from_pretrained(final_dir).to(device)   # a plain model, also when trained with LoRA
show_generations(model, tokenizer, picks, device)
```